# Notebook 09: Backdoor Detection

## Why This Matters
You received a pretrained model — from a public hub, a vendor, or a contractor. You didn't train it. You don't have the training data. Can you determine whether it has a backdoor before deploying it?

This notebook covers the three most important detection approaches: Neural Cleanse (reverse-engineer a trigger), STRIP (detect triggered inputs at inference time), and activation clustering (find poisoned examples by their internal representations). These are the basis of the `model-scan` tool built in notebook 19.

## Structure
1. The detection problem: what you have and what you don't
2. Neural Cleanse: trigger reverse-engineering (Wang et al., 2019)
3. STRIP: runtime detection of triggered inputs (Gao et al., 2019)
4. Activation Clustering: find poisoned training examples (Chen et al., 2019)
5. Comparing detection approaches
6. Limitations and adaptive attacks against detection

## What You'll Learn
- Why the minimum-norm trigger is a signature of backdoor presence
- How STRIP exploits the rigidity of triggered predictions
- Why backdoored and clean examples cluster differently in activation space
- Which detection approach to use when, and what each requires

## Background

### The detection setting

The defender has:
- A trained model (weights accessible — white-box)
- A small clean dataset (a few hundred examples, correctly labeled)
- Optionally: the training data (for activation clustering)

The defender does NOT have:
- Knowledge of whether a backdoor exists
- The trigger pattern (that's what we're trying to find)
- Knowledge of the target class

This asymmetry defines the detection challenge. The attacker has designed the trigger; the defender must find it without knowing what they're looking for.

### Neural Cleanse (Wang et al., IEEE S&P 2019)

Neural Cleanse's core insight: for a backdoored model, there exists a small perturbation that causes any input to be classified as the target class. This perturbation (the reverse-engineered trigger) will be much smaller (in $L_1$ norm) than the perturbations needed to "convert" inputs to non-backdoored classes.

The algorithm:
1. For each candidate target class $t$:
   - Solve: $\min_\delta \|\delta\|_1 + \lambda \cdot \mathbb{E}_x[\mathcal{L}(f(x \odot (1-m) + \delta \odot m), t)]$
   - where $m$ is a learned mask that concentrates the trigger spatially
2. Compute the $L_1$ norm of the reverse-engineered trigger for each class
3. Apply the **anomaly index**: the class with anomalously small trigger norm is the backdoor target
4. If the anomaly index exceeds a threshold, declare the model backdoored

The anomaly index uses the median absolute deviation (MAD) to detect outliers in the trigger norm distribution — robust to the fact that we don't know how many classes are backdoored.

### STRIP (Gao et al., ACSAC 2019)

STRIP (STRong Intentional Perturbation) is a runtime detection approach — it operates at inference time on individual inputs, not on the model.

The insight: a backdoored input's prediction is **rigid** — it will be classified as the target class regardless of what other content is overlaid. A clean input's prediction is **fragile** — if you blend it with random other images, the prediction changes.

Algorithm:
1. For each incoming input $x$, generate $N$ perturbed versions: $x_i' = \alpha \cdot x + (1-\alpha) \cdot z_i$ where $z_i$ are random reference images
2. Compute the prediction entropy over the $N$ versions
3. Triggered inputs → low entropy (always predict target class)
4. Clean inputs → high entropy (predictions vary with blended content)
5. Flag inputs with entropy below a threshold

STRIP requires no model weights, no training data — only the ability to query the model, making it suitable for black-box deployment scenarios.

### Activation Clustering (Chen et al., 2019)

Activation clustering (AC) exploits the observation that backdoored training examples have a distinctive internal representation. When you extract penultimate-layer activations for all training examples of the target class, you'll see two clusters: one for genuinely target-class examples, and one for poisoned examples (other classes with trigger stamps).

Algorithm:
1. Collect penultimate-layer activations for all training examples of each class
2. Apply dimensionality reduction (PCA or UMAP)
3. Cluster with k-means (k=2 per class)
4. If a class has a small cluster that is suspiciously distinct, flag it as potentially poisoned
5. Inspect the flagged cluster to verify

AC requires access to the training data — or at least a representative labeled dataset. It's the most informative detection method (it identifies the poisoned examples, not just flags the model) but also the most data-hungry.

In [ ]:
%pip install -q torch torchvision numpy matplotlib scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

device = torch.device('mps' if torch.backends.mps.is_available() else
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))
])
train_data = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=256, shuffle=False)

In [ ]:
# Rebuild backdoored model from notebook 08
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(0.25)

    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = F.relu(self.fc1(x.flatten(1))); x = self.drop(x)
        return self.fc2(x)

    def get_activations(self, x):
        """Return penultimate layer activations (before fc2)."""
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = F.relu(self.fc1(x.flatten(1)))
        return x  # (B, 128)


def apply_patch_trigger(x, patch_size=4):
    x = x.clone(); H, W = x.shape[-2], x.shape[-1]
    x[..., H-patch_size:H, W-patch_size:W] = x.max()
    return x


class PoisonedDataset(Dataset):
    def __init__(self, base, trigger_fn, target, rate, seed=42):
        self.base = base; self.trigger = trigger_fn; self.target = target
        rng = np.random.RandomState(seed)
        labels = [base[i][1] for i in range(len(base))]
        non_target = [i for i, y in enumerate(labels) if y != target]
        n = int(len(non_target) * rate)
        self.poison_set = set(rng.choice(non_target, n, replace=False).tolist())
    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        x, y = self.base[idx]
        if idx in self.poison_set: x = self.trigger(x); y = self.target
        return x, y


TARGET_CLASS = 0
poisoned_train = PoisonedDataset(train_data, apply_patch_trigger, TARGET_CLASS, 0.05)
poisoned_loader = DataLoader(poisoned_train, batch_size=256, shuffle=True)

# Train clean and backdoored models
def train(loader, epochs=8, seed=42):
    torch.manual_seed(seed)
    m = MnistCNN().to(device)
    o = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(epochs):
        m.train()
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            o.zero_grad(); F.cross_entropy(m(x), y).backward(); o.step()
    return m

print("Training clean model...")
model_clean    = train(train_loader)
print("Training backdoored model...")
model_backdoor = train(poisoned_loader)
print("Done.")

## 1. Neural Cleanse: Trigger Reverse-Engineering

In [ ]:
def neural_cleanse_per_class(
    model: nn.Module,
    clean_loader: DataLoader,
    target_class: int,
    n_steps: int = 200,
    lr: float = 0.1,
    lambda_l1: float = 1e-2,
    n_batches: int = 5,
) -> tuple:
    """
    Neural Cleanse: reverse-engineer the minimum trigger for a candidate target class.

    Optimizes a mask m and pattern delta such that:
      x_triggered = x * (1 - m) + delta * m
    is classified as target_class with high confidence.

    Returns the L1 norm of the reverse-engineered trigger (smaller = more suspicious).
    """
    model.eval()

    # Learnable mask [0,1] and pattern [input range]
    mask    = torch.zeros(1, 1, 28, 28, device=device, requires_grad=True)
    pattern = torch.zeros(1, 1, 28, 28, device=device, requires_grad=True)

    optimizer = torch.optim.Adam([mask, pattern], lr=lr)
    target_t  = torch.tensor([target_class], device=device)

    # Collect a small set of clean inputs
    clean_batch = []
    for i, (x, _) in enumerate(clean_loader):
        if i >= n_batches: break
        clean_batch.append(x)
    X_clean = torch.cat(clean_batch).to(device)

    for step in range(n_steps):
        optimizer.zero_grad()

        m = torch.sigmoid(mask)      # constrain to [0, 1]
        d = torch.tanh(pattern) * 3  # constrain to bounded range

        # Apply mask: blend clean input with the trigger pattern
        idx = torch.randperm(len(X_clean))[:32]
        x_batch   = X_clean[idx]
        x_trig    = x_batch * (1 - m) + d * m

        # Loss: high confidence for target class + small mask (L1)
        target_batch = target_t.expand(len(x_batch))
        cls_loss = F.cross_entropy(model(x_trig), target_batch)
        l1_loss  = m.abs().sum()

        loss = cls_loss + lambda_l1 * l1_loss
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        m_final = torch.sigmoid(mask)
        trigger_norm = m_final.abs().sum().item()

    return trigger_norm, torch.sigmoid(mask).detach(), torch.tanh(pattern).detach()


# Run Neural Cleanse across all classes for clean and backdoored models
def run_neural_cleanse(model, clean_loader, model_name=""):
    print(f"\nNeural Cleanse: {model_name}")
    norms = []
    masks = []
    for c in range(10):
        norm, mask, pattern = neural_cleanse_per_class(model, clean_loader, c, n_steps=150, n_batches=3)
        norms.append(norm)
        masks.append(mask)
        print(f"  Class {c}: trigger L1 norm = {norm:.3f}")

    # Anomaly index: MAD-based outlier detection
    norms_arr = np.array(norms)
    median = np.median(norms_arr)
    mad    = np.median(np.abs(norms_arr - median))
    anomaly_indices = np.abs(norms_arr - median) / (mad + 1e-8)

    print(f"\n  Anomaly indices (higher = more outlier):")
    flagged = []
    for c in range(10):
        flag = " ← FLAGGED" if anomaly_indices[c] > 2.0 else ""
        print(f"  Class {c}: {anomaly_indices[c]:.2f}{flag}")
        if anomaly_indices[c] > 2.0: flagged.append(c)

    backdoor_detected = len(flagged) > 0
    print(f"\n  Backdoor detected: {backdoor_detected}")
    if flagged:
        print(f"  Suspected target class(es): {flagged}")
    return norms, anomaly_indices, masks, backdoor_detected


small_clean_loader = DataLoader(torch.utils.data.Subset(test_data, range(500)), batch_size=64)

norms_clean, ai_clean, masks_clean, detected_clean = run_neural_cleanse(
    model_clean, small_clean_loader, "Clean model"
)
norms_bd, ai_bd, masks_bd, detected_bd = run_neural_cleanse(
    model_backdoor, small_clean_loader, "Backdoored model"
)

In [ ]:
# Visualize trigger norms and anomaly indices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

x_pos = np.arange(10)
axes[0].bar(x_pos - 0.2, norms_clean, 0.35, label='Clean model', color='steelblue', alpha=0.8)
axes[0].bar(x_pos + 0.2, norms_bd, 0.35, label='Backdoored model', color='darkorange', alpha=0.8)
axes[0].set_xlabel('Class'); axes[0].set_ylabel('Reverse-engineered trigger L1 norm')
axes[0].set_title('Neural Cleanse: Trigger Norm per Class\n(low norm = suspicious)')
axes[0].set_xticks(x_pos); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].bar(x_pos, ai_bd, color=['red' if ai > 2 else 'steelblue' for ai in ai_bd], alpha=0.8)
axes[1].axhline(2.0, color='red', linestyle='--', label='Detection threshold (2.0)')
axes[1].set_xlabel('Class'); axes[1].set_ylabel('Anomaly index')
axes[1].set_title('Neural Cleanse: Anomaly Index (Backdoored model)\n(red bars = flagged classes)')
axes[1].set_xticks(x_pos); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('neural_cleanse.png', bbox_inches='tight')
plt.show()

## 2. STRIP: Runtime Triggered Input Detection

In [ ]:
def strip_entropy(model, x, reference_images, n_perturb=20, alpha=0.5):
    """
    STRIP: compute prediction entropy of blended inputs.

    Triggered inputs → always predict target class → low entropy
    Clean inputs → prediction changes with blend → high entropy
    """
    model.eval()
    entropies = []

    with torch.no_grad():
        x_t = x.unsqueeze(0).to(device)
        # Sample random reference images and blend
        idx = torch.randperm(len(reference_images))[:n_perturb]
        for i in idx:
            ref = reference_images[i][0].unsqueeze(0).to(device)
            x_blend = alpha * x_t + (1 - alpha) * ref
            probs = F.softmax(model(x_blend), dim=1)
            entropy = -(probs * torch.log(probs + 1e-10)).sum().item()
            entropies.append(entropy)

    return np.mean(entropies)


# Compute STRIP entropy for clean and triggered test examples
reference_set = torch.utils.data.Subset(test_data, range(200))  # reference pool
n_eval = 100  # evaluate on 100 examples

clean_entropies     = []
triggered_entropies = []

print("Computing STRIP entropy for clean vs triggered inputs...")
for i in range(n_eval):
    x_clean, y = test_data[i]
    x_trig = apply_patch_trigger(x_clean)

    e_clean   = strip_entropy(model_backdoor, x_clean, reference_set)
    e_trig    = strip_entropy(model_backdoor, x_trig,  reference_set)

    clean_entropies.append(e_clean)
    triggered_entropies.append(e_trig)

# Threshold: the midpoint between mean entropies
threshold = (np.mean(clean_entropies) + np.mean(triggered_entropies)) / 2

tp = sum(e < threshold for e in triggered_entropies)  # triggered correctly detected
fp = sum(e < threshold for e in clean_entropies)       # clean incorrectly flagged

print(f"\nSTRIP Results (backdoored model, threshold={threshold:.3f}):")
print(f"  Mean entropy — clean:     {np.mean(clean_entropies):.4f}")
print(f"  Mean entropy — triggered: {np.mean(triggered_entropies):.4f}")
print(f"  Detection rate (TPR):     {tp/n_eval:.4f}")
print(f"  False positive rate:      {fp/n_eval:.4f}")
print()

# Visualize entropy distributions
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(clean_entropies,     bins=30, alpha=0.6, color='steelblue', label='Clean inputs', density=True)
ax.hist(triggered_entropies, bins=30, alpha=0.6, color='red', label='Triggered inputs', density=True)
ax.axvline(threshold, color='black', linestyle='--', label=f'Threshold = {threshold:.3f}')
ax.set_xlabel('Prediction entropy under perturbation')
ax.set_ylabel('Density')
ax.set_title('STRIP Detection: Entropy Distribution\n(low entropy = triggered)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('strip_detection.png', bbox_inches='tight')
plt.show()

## 3. Activation Clustering

In [ ]:
def extract_activations(model, loader, device, n_batches=None):
    """Extract penultimate-layer activations and labels."""
    model.eval()
    acts, labels = [], []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if n_batches and i >= n_batches: break
            x = x.to(device)
            acts.append(model.get_activations(x).cpu().numpy())
            labels.append(y.numpy())
    return np.vstack(acts), np.concatenate(labels)


def activation_clustering_detection(model, poisoned_loader, clean_loader, device):
    """
    Activation Clustering (Chen et al., 2019):
    Extract activations → PCA → k-means(k=2) per class → silhouette score.
    High silhouette = two distinct clusters = possible backdoor.
    """
    print("Extracting activations from poisoned training data...")
    acts, labels = extract_activations(model, poisoned_loader, device, n_batches=30)

    print(f"Activations shape: {acts.shape}")

    # PCA to 10 dimensions for clustering
    pca = PCA(n_components=10)
    acts_pca = pca.fit_transform(acts)

    results = {}
    for c in range(10):
        mask = labels == c
        if mask.sum() < 10: continue

        acts_c = acts_pca[mask]

        # k-means with k=2
        km = KMeans(n_clusters=2, random_state=42, n_init=10)
        cluster_labels = km.fit_predict(acts_c)

        # Silhouette score: how well separated are the clusters?
        if len(np.unique(cluster_labels)) > 1:
            sil = silhouette_score(acts_c, cluster_labels)
        else:
            sil = 0.0

        # Cluster size ratio: a small cluster of poisoned examples
        counts = np.bincount(cluster_labels)
        small_cluster_frac = min(counts) / sum(counts)

        results[c] = {
            'silhouette': sil,
            'small_cluster_frac': small_cluster_frac,
            'n_total': mask.sum(),
        }

    print(f"\n{'Class':>6} {'Silhouette':>12} {'Small cluster %':>16} {'Suspicious':>12}")
    print("-" * 52)
    for c, r in results.items():
        suspicious = r['silhouette'] > 0.3 and r['small_cluster_frac'] < 0.3
        print(f"{c:>6} {r['silhouette']:>12.4f} {r['small_cluster_frac']:>16.4f} {'YES' if suspicious else '':>12}")

    return results, acts_pca, labels


print("Activation clustering on BACKDOORED model:")
ac_results, acts_pca, ac_labels = activation_clustering_detection(
    model_backdoor, poisoned_loader, test_loader, device
)

In [ ]:
# Visualize: activations for class 0 (the backdoor target)
# Should show two clusters: genuine 0s and poisoned non-0s
class0_mask = ac_labels == 0
acts_class0 = acts_pca[class0_mask]

pca_2d = PCA(n_components=2)
acts_2d = pca_2d.fit_transform(acts_class0)

km2 = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels_0 = km2.fit_predict(acts_class0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['steelblue', 'darkorange']
for cluster in [0, 1]:
    mask_c = cluster_labels_0 == cluster
    axes[0].scatter(acts_2d[mask_c, 0], acts_2d[mask_c, 1],
                    c=colors[cluster], alpha=0.5, s=10,
                    label=f'Cluster {cluster} (n={mask_c.sum()})')
axes[0].set_title('Activation Clustering: Class 0\n(backdoor target class)')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Compare with class 1 (no backdoor)
class1_mask = ac_labels == 1
acts_class1 = pca_2d.transform(acts_pca[class1_mask])
km1 = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels_1 = km1.fit_predict(acts_pca[class1_mask])

for cluster in [0, 1]:
    mask_c = cluster_labels_1 == cluster
    axes[1].scatter(acts_class1[mask_c, 0], acts_class1[mask_c, 1],
                    c=colors[cluster], alpha=0.5, s=10,
                    label=f'Cluster {cluster} (n={mask_c.sum()})')
axes[1].set_title('Activation Clustering: Class 1\n(clean class — no backdoor)')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('activation_clustering.png', bbox_inches='tight')
plt.show()

print("Class 0 (backdoor target): look for two well-separated clusters")
print("Class 1 (clean): clusters should be less separated")

## 4. Detection Method Comparison

| Method | Data required | Model access | What it finds | Limitations |
|--------|--------------|--------------|----------------|-------------|
| Neural Cleanse | Small clean set | White-box (gradients) | Whether model is backdoored; suspected target class | Slow (optimization per class); fails against some adaptive triggers |
| STRIP | Reference images | Black-box (queries) | Individual triggered inputs at runtime | Requires many queries per decision; higher FPR at low entropy threshold |
| Activation Clustering | Training data | White-box (activations) | Which training examples are poisoned | Requires training data access; needs k-means to separate |

**When to use which:**
- **Received a pretrained model, no training data**: Neural Cleanse first, then STRIP for deployment
- **Have training data, suspect poisoning**: Activation Clustering to identify and remove poisoned examples
- **Deployed model, need runtime protection**: STRIP on every inference request
- **All three**: belt-and-suspenders for high-stakes deployments

## Summary

**Three complementary detection approaches:**
1. **Neural Cleanse**: reverse-engineer the minimum trigger per class; flag classes with anomalously small triggers. White-box, requires optimization. The anomaly index is the key statistic.
2. **STRIP**: at inference time, blend inputs with random images; triggered inputs show low prediction entropy. Black-box, fast, runtime-deployable.
3. **Activation Clustering**: extract penultimate-layer representations; poisoned examples cluster separately from genuine ones. Requires training data access; most interpretable (identifies the poisoned examples).

**None of these is perfect** — adaptive attacks that minimize trigger norm, maximize entropy under blending, or minimize activation distance can evade each. Defense requires combining approaches and staying current with attack literature.

**Next:** Notebook 10 covers data poisoning — broader than backdoor attacks, covering label flipping, gradient-based poisoning, and clean-label attacks against the model's general accuracy (not just targeted misclassification).